<a href="https://colab.research.google.com/github/zhesun0304/ECON3916/blob/main/Lab13/Hedonic_Pricing_and_the_FWL_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 1: Environment Initialization and Baseline Bivariate Regression

import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Load the dataset from the GitHub raw link
url = "https://raw.githubusercontent.com/zhesun0304/ECON3916/main/data/Zillow_California_2026_Hedonic.csv"
df = pd.read_csv(url)

# Naive bivariate regression: Sale_Price on Property_Age only
naive_model = smf.ols("Sale_Price ~ Property_Age", data=df).fit()

print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params["Property_Age"])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Mon, 30 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        01:03:57   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [2]:
# Step 2: The Multivariate Model (Controlling for Distance_to_Tech_Hub)

multi_model = smf.ols("Sale_Price ~ Property_Age + Distance_to_Tech_Hub", data=df).fit()

print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params["Property_Age"])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Mon, 30 Mar 2026   Prob (F-statistic):               0.00
Time:                        01:09:09   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [4]:
# Step 3: FWL Theorem Manual Proof

# 3a: Partial out distance from Sale_Price
res_y_model = smf.ols("Sale_Price ~ Distance_to_Tech_Hub", data=df).fit()
df["Price_Residuals"] = res_y_model.resid

# 3b: Partial out distance from Property_Age
res_x_model = smf.ols("Property_Age ~ Distance_to_Tech_Hub", data=df).fit()
df["Age_Residuals"] = res_x_model.resid

# 3c: Regress residuals on residuals
# The "- 1" removes the intercept for exact FWL matching
fwl_model = smf.ols("Price_Residuals ~ Age_Residuals - 1", data=df).fit()

print("\nFWL Isolated Age Coefficient:", fwl_model.params["Age_Residuals"])


FWL Isolated Age Coefficient: -2063.129216802139


In [5]:
# Step 4: Epistemological Proof
# Compare the FWL coefficient to the multivariate coefficient

multi_age_coef = multi_model.params["Property_Age"]
fwl_age_coef = fwl_model.params["Age_Residuals"]

print("Multivariate Age Coefficient :", multi_age_coef)
print("FWL Isolated Age Coefficient :", fwl_age_coef)
print("Difference                  :", multi_age_coef - fwl_age_coef)

Multivariate Age Coefficient : -2063.1292168021246
FWL Isolated Age Coefficient : -2063.129216802139
Difference                  : 1.4551915228366852e-11


In [6]:
# Step 5: Interactive 3D Scatter + Fitted Regression Hyperplane

import numpy as np
import plotly.graph_objects as go

# ------------------------------------------------------------
# Assumption:
# - df already contains:
#     Property_Age, Distance_to_Tech_Hub, Sale_Price
# - multi_model is already fit from:
#     smf.ols("Sale_Price ~ Property_Age + Distance_to_Tech_Hub", data=df).fit()
# ------------------------------------------------------------

# Extract the estimated OLS coefficients directly from the statsmodels results object.
# statsmodels stores them in .params using the variable names from the formula.
beta_0 = multi_model.params["Intercept"]
beta_age = multi_model.params["Property_Age"]
beta_dist = multi_model.params["Distance_to_Tech_Hub"]

print("Intercept:", beta_0)
print("Property_Age coefficient:", beta_age)
print("Distance_to_Tech_Hub coefficient:", beta_dist)

# Create smooth ranges covering the observed support of the two regressors.
# These will define the x-axis and y-axis span of the fitted plane.
age_range = np.linspace(df["Property_Age"].min(), df["Property_Age"].max(), 50)
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(), df["Distance_to_Tech_Hub"].max(), 50)

# Generate a 2D meshgrid from the two 1D ranges.
# meshgrid turns the x and y ranges into coordinate matrices so we can compute
# the predicted z-value (Sale_Price) at every combination of age and distance.
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Compute the fitted regression hyperplane over the full grid.
# This uses the OLS equation:
#   predicted_price = intercept
#                   + beta_age  * Property_Age
#                   + beta_dist * Distance_to_Tech_Hub
# Because the model is linear in both regressors, the fitted surface is a plane.
price_grid = beta_0 + beta_age * age_grid + beta_dist * dist_grid

# Optional: fitted values for the observed points
df["Predicted_Price"] = multi_model.fittedvalues

# 3D scatter of observed data points
scatter = go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    name="Observed Sales",
    marker=dict(
        size=4,
        opacity=0.75,
        color=df["Sale_Price"],
        colorscale="Viridis",
        colorbar=dict(title="Sale Price")
    ),
    hovertemplate=(
        "Property Age: %{x:.2f}<br>"
        "Distance to Tech Hub: %{y:.2f}<br>"
        "Actual Sale Price: %{z:,.2f}<extra></extra>"
    )
)

# 3D fitted OLS plane
surface = go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_grid,
    name="Fitted OLS Plane",
    opacity=0.6,
    colorscale="Blues",
    showscale=False,
    hovertemplate=(
        "Property Age: %{x:.2f}<br>"
        "Distance to Tech Hub: %{y:.2f}<br>"
        "Predicted Sale Price: %{z:,.2f}<extra></extra>"
    )
)

# Combine into a single interactive figure
fig = go.Figure(data=[surface, scatter])

fig.update_layout(
    title="3D Hedonic Pricing Model: Observed Data and Fitted Regression Hyperplane",
    scene=dict(
        xaxis_title="Property_Age",
        yaxis_title="Distance_to_Tech_Hub",
        zaxis_title="Sale_Price",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.1))
    ),
    width=950,
    height=750,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()

Intercept: 1202971.274417617
Property_Age coefficient: -2063.1292168021246
Distance_to_Tech_Hub coefficient: -7964.244819430518
